# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [1]:
!pip install evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00


In [2]:
import os
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
import evaluate


2026-03-01 12:18:11.793083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772367491.980226      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772367492.034480      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772367492.487148      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772367492.487189      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772367492.487192      23 computation_placer.cc:177] computation placer alr

In [3]:
class Config:
    # Akkadian transliteration contains a lot of noise and many unknown words, so
    # ByT5, which processes text at the character (byte) level rather than the word level, is the strongest choice.
    MODEL_NAME = "google/byt5-small" 

    # ByT5 tends to produce longer token sequences, but 512 tokens is enough at the sentence level.
    MAX_LENGTH = 512

    BATCH_SIZE = 8       # Adjust depending on GPU memory (on a P100 you can usually go with 8–16).
    EPOCHS = 20
    LEARNING_RATE = 2e-4

    # Where a single trained model would be saved (kept for compatibility)
    OUTPUT_DIR = "./byt5-akkadian-model"

    # ============ CV (5-fold) ============
    CV_N_SPLITS = 5
    CV_SEED = 42
    USE_GROUP_KFOLD = True
    SAVE_EACH_FOLD_MODEL = True

    # Save fold artifacts under OUTPUT_DIR
    FOLD_OUTPUT_ROOT = f"{OUTPUT_DIR}/cv5"
    REPORT_CSV = "cv_results.csv"


In [4]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [5]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
train_df = pd.read_csv(f"{INPUT_DIR}/train.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

In [6]:
print(f"Original Train Data: {len(train_df)} docs")

Original Train Data: 1561 docs


In [7]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [8]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

# Prepare 5-fold CV splits.
cv_df = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(cv_df.columns)

groups = cv_df["oare_id"].values
X_idx = np.arange(len(cv_df))

splitter = GroupKFold(n_splits=Config.CV_N_SPLITS)
fold_splits = list(splitter.split(X_idx, y=None, groups=groups))

print(f"Prepared {len(fold_splits)} folds (GroupKFold by oare_id)")


Expanded Train Data: 1561 sentences (Alignment applied)
Prepared 5 folds (GroupKFold by oare_id)


In [9]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX = "translate Akkadian to English: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})


def normalize_transliteration(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # Unify common gap markers to a single token for better pattern consistency.
    text = text.replace("…", " <gap> ")
    text = re.sub(r"\b[xX]{1,}\b", " <gap> ", text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_function(examples):
    normalized_src = [normalize_transliteration(ex) for ex in examples["transliteration"]]
    inputs = [PREFIX + ex for ex in normalized_src]
    targets = [str(ex) for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [10]:
# ==========================================
# 4. 5-fold CV training (fine-tuning)
# ==========================================

# Metrics: competition uses geo mean of BLEU and chrF++ (corpus-level)
chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("sacrebleu")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Sometimes Seq2SeqTrainer can return logits; convert them to token ids.
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    # ByT5 decoding can crash if ids are negative/out of vocab.
    preds = np.asarray(preds)
    preds = preds.astype(np.int64, copy=False)
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    refs = [[x] for x in decoded_labels]
    chrf = chrf_metric.compute(predictions=decoded_preds, references=refs)["score"]
    bleu = bleu_metric.compute(predictions=decoded_preds, references=refs)["score"]

    geo_mean = (bleu * chrf) ** 0.5 if (bleu > 0 and chrf > 0) else 0.0

    return {"chrf": chrf, "bleu": bleu, "geo_mean": geo_mean}


# Build a single HF dataset and slice by fold indices.
ds_all = Dataset.from_pandas(cv_df)

fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print("" + "=" * 60)
    print(f"FOLD {fold}/{Config.CV_N_SPLITS - 1}")
    print("=" * 60)

    ds_train = ds_all.select(tr_idx.tolist())
    ds_val = ds_all.select(va_idx.tolist())

    # Tokenize inside fold
    tokenized_train = ds_train.map(
        preprocess_function,
        batched=True,
        remove_columns=ds_train.column_names,
    )
    tokenized_val = ds_val.map(
        preprocess_function,
        batched=True,
        remove_columns=ds_val.column_names,
    )

    # Fresh model per fold
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    fold_out = f"{Config.FOLD_OUTPUT_ROOT}/fold_{fold}"
    fold_model_out = f"{fold_out}/model"
    os.makedirs(fold_out, exist_ok=True)

    args = Seq2SeqTrainingArguments(
        output_dir=fold_out,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=Config.LEARNING_RATE,

        # === Key fixes ===
        fp16=False,                    # FP16 can be unstable for some setups; keep FP32 by default.
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        generation_max_length=Config.MAX_LENGTH,  # Crucial for ByT5; avoid too-short defaults
        generation_num_beams=2,
        # ==================

        weight_decay=0.01,
        num_train_epochs=Config.EPOCHS,
        predict_with_generate=True,
        logging_strategy="epoch",
        logging_steps=10,
        disable_tqdm=True,
        report_to="none",
        load_best_model_at_end=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    print("Starting Training (fold CV, FP32 mode)...")
    trainer.train()

    eval_metrics = trainer.evaluate()
    fold_result = {
        "fold": int(fold),
        "n_train": int(len(tokenized_train)),
        "n_val": int(len(tokenized_val)),
        "geo_mean": float(eval_metrics.get("eval_geo_mean", float("nan"))),
        "bleu": float(eval_metrics.get("eval_bleu", float("nan"))),
        "chrf": float(eval_metrics.get("eval_chrf", float("nan"))),
        "fold_model_path": fold_model_out,
    }
    fold_metrics.append(fold_result)

    print("Fold metrics:", fold_result)

    if Config.SAVE_EACH_FOLD_MODEL:
        trainer.save_model(fold_model_out)
        tokenizer.save_pretrained(fold_model_out)
        print(f"Saved fold model to: {fold_model_out}")

    # Explicit cleanup to avoid memory/disk pressure across folds
    del trainer, model, data_collator, tokenized_train, tokenized_val, ds_train, ds_val
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summarize CV
cv_results = pd.DataFrame(fold_metrics)
report_path = f"{Config.FOLD_OUTPUT_ROOT}/{Config.REPORT_CSV}"
os.makedirs(Config.FOLD_OUTPUT_ROOT, exist_ok=True)
cv_results.to_csv(report_path, index=False)

print("" + "=" * 60)
print("5-FOLD CV SUMMARY")
print("=" * 60)
print(cv_results[["fold", "n_train", "n_val", "geo_mean", "bleu", "chrf", "fold_model_path"]])

mean_score = cv_results["geo_mean"].mean()
std_score = cv_results["geo_mean"].std(ddof=1)
print(f"geo_mean: mean={mean_score:.4f}, std={std_score:.4f}")

best_i = int(cv_results["geo_mean"].idxmax())
print(f"Best fold index: {best_i}")
print(f"Best fold model path: {cv_results.loc[best_i, 'fold_model_path']}")
print(f"Saved CV report CSV to: {report_path}")


FOLD 0/4


Map:   0%|          | 0/1248 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipykernel_23/3338663462.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training (fold CV, FP32 mode)...
{'loss': 1.4677, 'grad_norm': 1.3645237684249878, 'learning_rate': 0.00019006410256410257, 'epoch': 1.0}
{'eval_loss': 0.8343082070350647, 'eval_chrf': 12.180464045893194, 'eval_bleu': 1.4215698552515015, 'eval_geo_mean': 4.161175376094657, 'eval_runtime': 395.1752, 'eval_samples_per_second': 0.792, 'eval_steps_per_second': 0.2, 'epoch': 1.0}
{'loss': 0.8898, 'grad_norm': 0.5359102487564087, 'learning_rate': 0.00018006410256410257, 'epoch': 2.0}
{'eval_loss': 0.707516074180603, 'eval_chrf': 22.507847833348794, 'eval_bleu': 4.956781132295362, 'eval_geo_mean': 10.56250327663468, 'eval_runtime': 396.5725, 'eval_samples_per_second': 0.789, 'eval_steps_per_second': 0.199, 'epoch': 2.0}
{'loss': 0.7645, 'grad_norm': 0.46310392022132874, 'learning_rate': 0.00017006410256410257, 'epoch': 3.0}
{'eval_loss': 0.6568611264228821, 'eval_chrf': 29.082787704102408, 'eval_bleu': 9.494795843633206, 'eval_geo_mean': 16.617314217832508, 'eval_runtime': 394.8105, 

ValueError: chr() arg not in range(0x110000)

In [ ]:
# --- Notes ---
# Fold models are saved under:
#   ./byt5-akkadian-model/cv5/fold_{k}/model
# The CV summary is saved to:
#   ./byt5-akkadian-model/cv5/cv_results.csv
